In [17]:
import pandas as pd
import numpy as np
import os
import cv2
import tensorflow
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, AveragePooling2D, Flatten, Dropout

import warnings
warnings.filterwarnings("ignore")

In [2]:
Rice_Image_Dataset = r"E:\Rutuja_datascience_2026\Rice_Classification\Rice_Image_Dataset"

In [3]:
classes = sorted(os.listdir(Rice_Image_Dataset))
print(classes)

['Arborio', 'Basmati', 'Ipsala', 'Jasmine', 'karacadag']


### Data Processing

In [4]:
for cls in classes:
    folder_path = os.path.join("Rice_Image_Dataset", cls)
    print(f"{cls}: {len(os.listdir(folder_path))}")

Arborio: 200
Basmati: 200
Ipsala: 200
Jasmine: 200
karacadag: 200


In [12]:
image_size=(64,64)
data_generator = ImageDataGenerator(rescale = 1./255, validation_split = 0.2)
train_generator = data_generator.flow_from_directory("Rice_Image_Dataset/", target_size = image_size, batch_size = 32, 
                                                    class_mode = "categorical", subset = "training", shuffle = True)

validation_generator = data_generator.flow_from_directory("Rice_Image_Dataset/", target_size = image_size, batch_size = 32, 
                                                    class_mode = "categorical", subset = "validation", shuffle = False)


Found 800 images belonging to 5 classes.
Found 200 images belonging to 5 classes.


In [7]:
print(train_generator.class_indices)

{'Arborio': 0, 'Basmati': 1, 'Ipsala': 2, 'Jasmine': 3, 'karacadag': 4}


In [8]:
image, labels = next(train_generator)
print(image.shape)
print(labels.shape)

(32, 64, 64, 3)
(32, 5)


### Model Building

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.layers import Dense, Dropout

model = Sequential()

model.add(Conv2D(32,(3,3),activation="relu",input_shape=(224,224,3)))
model.add(MaxPooling2D((2,2)))

model.add(Conv2D(64,(3,3),activation="relu"))
model.add(MaxPooling2D((2,2)))

model.add(GlobalAveragePooling2D())

model.add(Dense(128,activation="relu"))
model.add(Dropout(0.5))

model.add(Dense(5,activation="softmax"))

In [14]:
model.compile(optimizer="adam",loss="categorical_crossentropy", metrics=["accuracy"])
history = model.fit(train_generator, validation_data = validation_generator, epochs = 10, verbose = 1)


Epoch 1/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 68ms/step - accuracy: 0.6637 - loss: 0.7778 - val_accuracy: 0.8200 - val_loss: 0.6628
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 59ms/step - accuracy: 0.6313 - loss: 0.7983 - val_accuracy: 0.7700 - val_loss: 0.6597
Epoch 3/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 61ms/step - accuracy: 0.6575 - loss: 0.7580 - val_accuracy: 0.7950 - val_loss: 0.6359
Epoch 4/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 59ms/step - accuracy: 0.6825 - loss: 0.7436 - val_accuracy: 0.8700 - val_loss: 0.6122
Epoch 5/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 64ms/step - accuracy: 0.6675 - loss: 0.7407 - val_accuracy: 0.8400 - val_loss: 0.6034
Epoch 6/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 63ms/step - accuracy: 0.6875 - loss: 0.7269 - val_accuracy: 0.8550 - val_loss: 0.5877
Epoch 7/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 61ms/step - accuracy: 0.6787 - loss: 0.7319 - val_accuracy: 0.8400 - val_loss: 0.5849
Epoch 8/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 58ms/step - accuracy: 0.7075 - loss: 0.6975 - val_accuracy: 0.8400 - v

In [18]:
image_path = r"E:\Rutuja_datascience_2026\Rice_Classification\Rice_Image_Dataset\karacadag\Karacadag (993).jpg"
img = image.load_img(image_path, target_size = image_size)
img_array = image.img_to_array(img)
img_array = img_array / 255.0
test_image = np.expand_dims(img_array, axis = 0)

In [19]:
prediction = model.predict(test_image)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step


In [20]:
predicted_index = np.argmax(prediction)
print(predicted_index)

0


In [22]:
class_name = list(train_generator.class_indices.keys())
predicted_class = class_name[predicted_index]
print("predicted rice type :", predicted_class)

predicted rice type : Arborio


In [23]:
confidence = np.max(prediction) * 100
print(f"Confidence : {confidence:.2f}%")

Confidence : 65.86%


In [24]:
from tensorflow.keras.preprocessing import image
import numpy as np

class_names = list(train_generator.class_indices.keys())

def predict_class(image_path):
    img = image.load_img(image_path, target_size=image_size)
    image_array = image.img_to_array(img) / 255.0
    test_array = np.expand_dims(image_array, axis=0)
    pred_prob = model.predict(test_array, verbose=0)
    pred_index = np.argmax(pred_prob)
    pred_class = class_names[pred_index]
    confidence = np.max(pred_prob) * 100
    
    print("Image Path       :", image_path)
    print("Predicted Class  :", pred_class)
    print(f"Confidence       : {confidence:.2f}%")

In [25]:
predict_class(r"E:\Rutuja_datascience_2026\Rice_Classification\Rice_Image_Dataset\Ipsala\Ipsala (994).jpg")

Image Path       : E:\Rutuja_datascience_2026\Rice_Classification\Rice_Image_Dataset\Ipsala\Ipsala (994).jpg
Predicted Class  : Ipsala
Confidence       : 100.00%


In [26]:
predict_class(r"E:\Rutuja_datascience_2026\Rice_Classification\Rice_Image_Dataset\Arborio\Arborio (988).jpg")

Image Path       : E:\Rutuja_datascience_2026\Rice_Classification\Rice_Image_Dataset\Arborio\Arborio (988).jpg
Predicted Class  : Arborio
Confidence       : 66.96%


In [28]:
model.save("rice_classification.keras")